## Embedding Models

We will use open source models from Hugging Face repository.

We use the AutoTokenizer library for this.
Autokenizer is as a universal charger for text tokenization.

We will look at how tokenization is done, what are embeddings, how they are calculated, what is cosine similarity and how it is used for semantic similarity


In [3]:
!pip install transformers==4.57.1 datasets==4.8.4 evaluate==0.4.6 accelerate==1.13.0 chromadb==1.5.8 llama-index-core==0.13.6 llama-index-llms-groq==0.5.0 groq==1.2.0 llama-index-embeddings-huggingface==0.6.0 sentence-transformers==5.4.1 llama-index-vector-stores-chroma

We load the model and put it in evaluation mode.

Eval mode means that the model is being used for inference purposes, and not training.
This means that some layers will be affected, and optimized for inference purposes. In our case, the drop out layer (the layer that turns some neurons off during training purposes for better optimized pathway)

In [4]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel


model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Put model in evaluation mode
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSdpaAttention(
          (dropout): Dropout(p=0.1, inplace=False)
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

We take some samples of text to demo what is tokenization and what happens after texts are tokenized.

In [5]:
# -----------------------------
# 2. Example texts
# -----------------------------
texts = [
    "I love machine learning and artificial intelligence.",
    "I enjoy studying AI and machine learning.",
    "The cat is sleeping on the sofa."
]

# -----------------------------
# 3. Tokenization demo
# -----------------------------
print("\n" + "="*60)
print("TOKENIZATION DEMO")
print("="*60)

sample_text = texts[0]
encoded_sample = tokenizer(sample_text, return_tensors="pt")

print(f"\nOriginal text:\n{sample_text}\n")

# print("Tokenizer output keys:")
# print(encoded_sample.keys())

print("\nInput IDs:")
print(encoded_sample["input_ids"])

# print("\nAttention Mask:")
# print(encoded_sample["attention_mask"])

tokens = tokenizer.convert_ids_to_tokens(encoded_sample["input_ids"][0])
print("\nTokens:")
print(tokens)

print("\nToken ID -> Token mapping:")
for token_id, token in zip(encoded_sample["input_ids"][0], tokens):
    print(f"{int(token_id):>6} -> {token}")




TOKENIZATION DEMO

Original text:
I love machine learning and artificial intelligence.


Input IDs:
tensor([[ 101, 1045, 2293, 3698, 4083, 1998, 7976, 4454, 1012,  102]])

Tokens:
['[CLS]', 'i', 'love', 'machine', 'learning', 'and', 'artificial', 'intelligence', '.', '[SEP]']

Token ID -> Token mapping:
   101 -> [CLS]
  1045 -> i
  2293 -> love
  3698 -> machine
  4083 -> learning
  1998 -> and
  7976 -> artificial
  4454 -> intelligence
  1012 -> .
   102 -> [SEP]


Now we will pass those tokens through the model and see how it changes.
We will check how many numbers it takes to represent (embed) 1 token.

We will see what do those numbers represent and what they look like.

We will also see of what an embedding vector is made of (the shape and size)

The last hidden state is basically a summary of what happened.

In [6]:
print("\n" + "="*60)
print("MODEL OUTPUT DEMO")
print("="*60)

with torch.no_grad():
    outputs = model(**encoded_sample)

last_hidden_state = outputs.last_hidden_state

print("\nShape of last_hidden_state:")
print(last_hidden_state.shape)
print("Meaning: [batch_size, sequence_length, hidden_size]")

# For DistilBERT:
# batch_size = 1
# sequence_length = number of tokens
# hidden_size = 768

print("\nFirst token embedding vector (first 10 values):")
print(last_hidden_state[0, 0, :10])

print("\nEmbedding size of one token:")
print(last_hidden_state[0, 0].shape)



MODEL OUTPUT DEMO

Shape of last_hidden_state:
torch.Size([1, 10, 768])
Meaning: [batch_size, sequence_length, hidden_size]

First token embedding vector (first 10 values):
tensor([ 0.0357,  0.1110, -0.3135, -0.2606, -0.1206, -0.0558,  0.2631,  0.7192,
        -0.0623, -0.3851])

Embedding size of one token:
torch.Size([768])


In the above, the hidden state [1,10,768] means that we passed in 1 sentence, it produced 10 tokens and each token is in size 768.

Now embeddings are produced at token level for models like BERT. Which means that every word in a sentence is represented by a contextual embedding.
But we are comparing sentences, not words. What do we do?

The concept of it is that each word embeddings is affected by its context (surroundings, like each words comunicate with each other to tell each what they represent and how they are related)

And if we combine in a way all those individual word embeddings, we can get an approximate sentence level embedding that will tell us what the sentence means.

We achieve that by an operation called pooling (combining all the embeddings to each other)

In [7]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

    summed = torch.sum(token_embeddings * input_mask_expanded, dim=1)
    summed_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)

    return summed / summed_mask

Attention mask is a number given to each token to the sentence. If 1, it means that the token should be considered during the embedding phase. If 0, (usually for padding tokens), it means that the token should be ignored during the embedding phase

Here what we are doing is basically : sum of all token embeddings/number of real tokens

In [8]:
# 6. Create embeddings for all texts
# -----------------------------
print("\n" + "="*60)
print("SENTENCE EMBEDDING DEMO")
print("="*60)

encoded_all = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")

with torch.no_grad():
    outputs_all = model(**encoded_all)

sentence_embeddings = mean_pooling(outputs_all, encoded_all["attention_mask"])

print("\nShape of sentence embeddings:")
print(sentence_embeddings.shape)
print("Meaning: [number_of_sentences, embedding_dimension]")

for i, emb in enumerate(sentence_embeddings):
    print(f"\nSentence {i+1}: {texts[i]}")
    print(f"Embedding shape: {emb.shape}")
    print(f"First 10 values: {emb[:10]}")



SENTENCE EMBEDDING DEMO

Shape of sentence embeddings:
torch.Size([3, 768])
Meaning: [number_of_sentences, embedding_dimension]

Sentence 1: I love machine learning and artificial intelligence.
Embedding shape: torch.Size([768])
First 10 values: tensor([ 0.2363,  0.2331, -0.2529,  0.1614,  0.4731, -0.2985, -0.0564,  0.6516,
        -0.0299, -0.3160])

Sentence 2: I enjoy studying AI and machine learning.
Embedding shape: torch.Size([768])
First 10 values: tensor([ 0.2100,  0.2989, -0.1584,  0.2110,  0.3641, -0.3902,  0.0006,  0.5982,
        -0.1437, -0.3627])

Sentence 3: The cat is sleeping on the sofa.
Embedding shape: torch.Size([768])
First 10 values: tensor([-0.0203, -0.0193,  0.0953,  0.1151,  0.4263, -0.3909, -0.1094,  0.5104,
        -0.2391, -0.3712])


Now we normalize all the embeddings.

Normalizing means rescaling all vectors so that its length becomes 1.
So we keep the direction, but standardize the magnitude.
This is useful because cosine similarity cares about angle/direction, not raw size.

p=2 here means we are using the Euclidean normalization (also known as L2 normalization)

In [9]:
normalized_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)


We now calculate the cosine similarity for each text

In [10]:
# -----------------------------
# 8. Cosine similarity demo
# -----------------------------
print("\n" + "="*60)
print("COSINE SIMILARITY DEMO")
print("="*60)

sim_1_2 = F.cosine_similarity(
    normalized_embeddings[0].unsqueeze(0),
    normalized_embeddings[1].unsqueeze(0)
)

sim_1_3 = F.cosine_similarity(
    normalized_embeddings[0].unsqueeze(0),
    normalized_embeddings[2].unsqueeze(0)
)

sim_2_3 = F.cosine_similarity(
    normalized_embeddings[1].unsqueeze(0),
    normalized_embeddings[2].unsqueeze(0)
)

print(f"\nText 1: {texts[0]}")
print(f"Text 2: {texts[1]}")
print(f"Cosine similarity (1,2): {sim_1_2.item():.4f}")

print(f"\nText 1: {texts[0]}")
print(f"Text 3: {texts[2]}")
print(f"Cosine similarity (1,3): {sim_1_3.item():.4f}")

print(f"\nText 2: {texts[1]}")
print(f"Text 3: {texts[2]}")
print(f"Cosine similarity (2,3): {sim_2_3.item():.4f}")



COSINE SIMILARITY DEMO

Text 1: I love machine learning and artificial intelligence.
Text 2: I enjoy studying AI and machine learning.
Cosine similarity (1,2): 0.9579

Text 1: I love machine learning and artificial intelligence.
Text 3: The cat is sleeping on the sofa.
Cosine similarity (1,3): 0.6593

Text 2: I enjoy studying AI and machine learning.
Text 3: The cat is sleeping on the sofa.
Cosine similarity (2,3): 0.6822


## Vector databases

Here we will use Chroma DB as our vector database.

We shall see the structure of ChromaDB, the embedding models, collections, metadata filtering, add data to our collection and then querying the vector db

In [11]:
import chromadb

client = chromadb.Client()

collection = client.create_collection(name="ai_collective_demo",configuration={
        "hnsw": {
            "space": "cosine"
        }
    }
                                      )

In [12]:
print("Class name:", collection._embedding_function.__class__.__name__)


Class name: DefaultEmbeddingFunction


By default, chroma db already has an inbuilt embedding model called all-MiniLM-L6-v2, which is of the family sentence transformers.

Now lets create some documents, with metadata and ids.

We will add them to the collection we created.

We pass in the ids, the documents and the metadata.
Chroma db will automatically use the embedding functions to convert the documents to embeddings.

In [13]:
documents = [
    """Machine learning can help businesses predict customer churn by analyzing historical behavior such as purchase frequency, support interactions, and account activity. By detecting early warning patterns, companies can intervene before customers leave and improve long-term retention.""",

    """Deep learning is widely used in computer vision for tasks such as image classification, object detection, and facial recognition. Neural networks learn hierarchical patterns from pixels, allowing systems to recognize complex visual structures with high accuracy.""",

    """Financial forecasting allows companies to estimate future revenue, costs, and cash flow using historical business data. These forecasts support budgeting, investment planning, and decision-making under uncertainty.""",

    """Revenue prediction models help organizations anticipate future sales trends by learning from previous transactions, seasonality, and market conditions. Accurate predictions can improve inventory planning and strategic resource allocation.""",

    """Natural language processing enables computers to understand and analyze human language in text and speech form. It is used in chatbots, sentiment analysis, translation systems, and document classification.""",

    """Recommendation systems suggest products, movies, or articles by learning from user preferences and past interactions. They are commonly used in e-commerce and streaming platforms to improve engagement and personalization.""",

    """Fraud detection systems use machine learning to identify unusual transaction behavior that may indicate financial crime. These systems often combine anomaly detection, classification models, and business rules to reduce risk.""",

    """Customer segmentation groups people into categories based on shared characteristics such as age, purchase behavior, and interests. Businesses use segmentation to create targeted marketing campaigns and more personalized customer experiences.""",

    """Time series forecasting is used to predict future values such as weekly sales, stock prices, or electricity demand. It relies on patterns over time, including trend, seasonality, and past fluctuations.""",

    """Search engines increasingly rely on embeddings and semantic retrieval to understand the meaning behind a user query rather than matching only exact keywords. This improves the relevance of search results, especially for paraphrased or ambiguous requests.""",

    """Cats are known for sleeping many hours each day, often alternating between deep rest and short periods of alertness. Their behavior is influenced by age, environment, hunting instincts, and energy conservation needs.""",

    """Dogs are social animals that often form strong bonds with humans and other dogs. Their behavior can include guarding, companionship, playfulness, and trainability depending on breed, upbringing, and environment.""",

    """Marine ecosystems are shaped by interactions between fish, coral reefs, sea plants, and water quality conditions. Changes in temperature, pollution, and acidity can significantly affect biodiversity and ecosystem stability.""",

    """Bird migration is a seasonal movement pattern driven by food availability, breeding cycles, and climate. Many bird species travel thousands of kilometers each year, relying on navigation cues such as stars, magnetic fields, and landmarks.""",

    """Renewable energy sources such as solar, wind, and hydroelectric power reduce dependence on fossil fuels and lower greenhouse gas emissions. Their adoption is increasing as governments and industries seek more sustainable energy systems.""",

    """Climate change affects agriculture through rising temperatures, shifting rainfall patterns, and more frequent extreme weather events. Farmers may need to adapt by changing crop selection, irrigation methods, and planting schedules.""",

    """Healthcare analytics uses data from hospitals, clinics, and wearable devices to improve patient care and operational efficiency. Predictive models can support early diagnosis, patient risk scoring, and resource planning.""",

    """Electronic health records store patient information such as diagnoses, treatments, laboratory results, and appointment history in digital form. Proper use of these systems can improve continuity of care and reduce administrative burden.""",

    """Cybersecurity involves protecting systems, networks, and data from unauthorized access, attacks, and disruptions. Common practices include encryption, authentication, monitoring, patch management, and employee awareness training.""",

    """Cloud computing allows businesses to access computing resources such as storage, servers, and machine learning services over the internet. It provides flexibility, scalability, and cost efficiency compared to maintaining all infrastructure on-premises."""
]

metadatas = [
    {"category": "AI", "year": 2025, "topic": "churn"},
    {"category": "AI", "year": 2024, "topic": "computer_vision"},
    {"category": "Finance", "year": 2025, "topic": "forecasting"},
    {"category": "Finance", "year": 2024, "topic": "revenue_prediction"},
    {"category": "AI", "year": 2025, "topic": "nlp"},
    {"category": "AI", "year": 2024, "topic": "recommendation_systems"},
    {"category": "Finance", "year": 2025, "topic": "fraud_detection"},
    {"category": "Business", "year": 2023, "topic": "customer_segmentation"},
    {"category": "Data", "year": 2025, "topic": "time_series"},
    {"category": "AI", "year": 2025, "topic": "semantic_search"},
    {"category": "Animals", "year": 2025, "topic": "cats"},
    {"category": "Animals", "year": 2023, "topic": "dogs"},
    {"category": "Animals", "year": 2024, "topic": "marine_ecosystems"},
    {"category": "Animals", "year": 2024, "topic": "birds"},
    {"category": "Environment", "year": 2025, "topic": "renewable_energy"},
    {"category": "Environment", "year": 2024, "topic": "climate_change"},
    {"category": "Healthcare", "year": 2025, "topic": "healthcare_analytics"},
    {"category": "Healthcare", "year": 2023, "topic": "ehr"},
    {"category": "Technology", "year": 2025, "topic": "cybersecurity"},
    {"category": "Technology", "year": 2024, "topic": "cloud_computing"},
]

ids = [f"doc{i}" for i in range(1, 21)]

collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas
)

Lets check what happened to the documents

In [14]:
# See how many records are in the collection
print("Count:", collection.count())

data = collection.get(include=["documents", "embeddings", "metadatas"])

for i in range(len(data["ids"])):
    print(f"ID: {data['ids'][i]}")
    print(f"Document: {data['documents'][i]}")
    print(f"Metadata: {data['metadatas'][i]}")
    print(f"Embedding length: {len(data['embeddings'][i])}")
    print(f"First 10 embedding values: {data['embeddings'][i][:10]}")
    print("-" * 50)

Count: 20
ID: doc1
Document: Machine learning can help businesses predict customer churn by analyzing historical behavior such as purchase frequency, support interactions, and account activity. By detecting early warning patterns, companies can intervene before customers leave and improve long-term retention.
Metadata: {'topic': 'churn', 'category': 'AI', 'year': 2025}
Embedding length: 384
First 10 embedding values: [-0.0406262  -0.0435742   0.03556292  0.00983215  0.03455004  0.04730873
  0.03819589 -0.05111566 -0.00208106 -0.10878033]
--------------------------------------------------
ID: doc2
Document: Deep learning is widely used in computer vision for tasks such as image classification, object detection, and facial recognition. Neural networks learn hierarchical patterns from pixels, allowing systems to recognize complex visual structures with high accuracy.
Metadata: {'category': 'AI', 'topic': 'computer_vision', 'year': 2024}
Embedding length: 384
First 10 embedding values: [-0

In [ ]:
# results = collection.query(
#     query_texts=["How can AI help understand meaning in search queries?"],
#     n_results=3,
#     include=["documents", "metadatas", "distances"]
# )

# query = "How can AI help understand meaning in search queries?"

# print("=" * 80)
# print("QUERY")
# print("=" * 80)
# print(query)

# print("\n" + "=" * 80)
# print("TOP MATCHES")
# print("=" * 80)

# for i in range(len(results["ids"][0])):
#     print(f"\nResult #{i+1}")
#     print("-" * 80)
#     print(f"ID        : {results['ids'][0][i]}")
#     print(f"Distance  : {results['distances'][0][i]:.4f}")
#     print(f"Metadata  : {results['metadatas'][0][i]}")
#     print(f"Document  :\n{results['documents'][0][i]}")

Now we try out some metadata filtering with it

In [15]:
results = collection.query(
    query_texts=["How do animals behave in nature?"],
    n_results=5,
    # where={"category": "Animals"},
    include=["documents", "metadatas", "distances"]
)


query = "How do animals behave in nature?"
print("=" * 80)
print("QUERY")
print("=" * 80)
print(query)

print("\n" + "=" * 80)
print("TOP MATCHES")
print("=" * 80)

for i in range(len(results["ids"][0])):
    print(f"\nResult #{i+1}")
    print("-" * 80)
    print(f"ID        : {results['ids'][0][i]}")
    print(f"Distance  : {results['distances'][0][i]:.4f}")
    print(f"Metadata  : {results['metadatas'][0][i]}")
    print(f"Document  :\n{results['documents'][0][i]}")

QUERY
How do animals behave in nature?

TOP MATCHES

Result #1
--------------------------------------------------------------------------------
ID        : doc12
Distance  : 0.5379
Metadata  : {'category': 'Animals', 'year': 2023, 'topic': 'dogs'}
Document  :
Dogs are social animals that often form strong bonds with humans and other dogs. Their behavior can include guarding, companionship, playfulness, and trainability depending on breed, upbringing, and environment.

Result #2
--------------------------------------------------------------------------------
ID        : doc13
Distance  : 0.6697
Metadata  : {'topic': 'marine_ecosystems', 'category': 'Animals', 'year': 2024}
Document  :
Marine ecosystems are shaped by interactions between fish, coral reefs, sea plants, and water quality conditions. Changes in temperature, pollution, and acidity can significantly affect biodiversity and ecosystem stability.

Result #3
------------------------------------------------------------------------

## RAG Demo

We shall be using llama index framework for this demo

llama index is a framework that enables us to orchestrate components seamlessly for RAG purposes, among many.

The concepts involved are Simple Directory that stores the documents,

In [16]:
import os
import chromadb

from llama_index.core import (
    Settings,
    SimpleDirectoryReader,
    VectorStoreIndex,
    StorageContext,
)
from llama_index.core.schema import MetadataMode
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore


In [17]:
# ---------------------------------------------------
# 1. Configure LLM + embedding model
# ---------------------------------------------------
groq_api_key = ""

llm = Groq(
    model="llama-3.3-70b-versatile",
    api_key=groq_api_key,
    temperature=0,
)
Settings.llm = llm

embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-large-en-v1.5",
    trust_remote_code=True,
)
Settings.embed_model = embed_model



In [1]:
!pip install llama-index-readers-file

In [18]:
# ---------------------------------------------------
# 2. Load your documents
# ---------------------------------------------------
reader = SimpleDirectoryReader(input_dir="./docs")
docs=reader.load_data()

In [20]:
print("RAW DOC:", docs[0].text[:1000])

RAW DOC: Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser ∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
ent

In [21]:
# ---------------------------------------------------
# 3. Create / connect to ChromaDB
#    Use PersistentClient if you want data saved on disk
# ---------------------------------------------------
chroma_client = chromadb.PersistentClient(path="./chroma_db")

chroma_collection = chroma_client.get_or_create_collection(
    name="mauritius_ai_strategy",configuration={
        "hnsw": {
            "space": "cosine"
        }
    }
)

# -----------------------------------------------

# 4. Wrap the Chroma collection in a LlamaIndex vector store
# ---------------------------------------------------
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

storage_context = StorageContext.from_defaults(vector_store=vector_store)

# ---------------------------------------------------
# 5. Build the index using Chroma as the backing store
# ---------------------------------------------------
index = VectorStoreIndex.from_documents(
    docs,
    storage_context=storage_context,
    show_progress=True,
)

Parsing nodes:   0%|          | 0/15 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/15 [00:00<?, ?it/s]

In [22]:
print("Count in Chroma:", chroma_collection.count())

data = chroma_collection.get(include=["documents", "metadatas", "embeddings"])


Count in Chroma: 15


In [23]:

for i in range(min(5, len(data["ids"]))):
    print(f"\nStored record #{i+1}")
    print("ID:", data["ids"][i])
    print("Metadata:", data["metadatas"][i])
    print("Document text:")
    print(data["documents"][i][:800])
    print("Embedding dim:", len(data["embeddings"][i]))
    print("First 10 embedding values:", data["embeddings"][i][:10])


Stored record #1
ID: ee0ff811-09c3-4107-b617-c8f43d061912
Metadata: {'creation_date': '2026-04-24', '_node_type': 'TextNode', 'page_label': '1', 'document_id': '3e773c63-f078-4653-be00-b7f6ff8a5107', '_node_content': '{"id_": "ee0ff811-09c3-4107-b617-c8f43d061912", "embedding": null, "metadata": {"page_label": "1", "file_name": "attention_is_all_you_need.pdf", "file_path": "/content/docs/attention_is_all_you_need.pdf", "file_type": "application/pdf", "file_size": 2215244, "creation_date": "2026-04-24", "last_modified_date": "2026-04-24"}, "excluded_embed_metadata_keys": ["file_name", "file_type", "file_size", "creation_date", "last_modified_date", "last_accessed_date"], "excluded_llm_metadata_keys": ["file_name", "file_type", "file_size", "creation_date", "last_modified_date", "last_accessed_date"], "relationships": {"1": {"node_id": "3e773c63-f078-4653-be00-b7f6ff8a5107", "node_type": "4", "metadata": {"page_label": "1", "file_name": "attention_is_all_you_need.pdf", "file_path": "/co

In [25]:
# ---------------------------------------------------
# 6. Create query engine
# ---------------------------------------------------
qe = index.as_query_engine(
    similarity_top_k=5,
    response_mode="tree_summarize",
    verbose=True,
)

# ---------------------------------------------------
# 7. Query
# ---------------------------------------------------
query = "what is the purpose of encoder and decoder in the transformer network?"
response = qe.query(query)

print("\n" + "=" * 100)
print("FINAL RESPONSE")
print("=" * 100)
print(str(response))

# ---------------------------------------------------
# 8. Print sources nicely
# ---------------------------------------------------
print("\n" + "=" * 100)
print("SOURCE NODES")
print("=" * 100)

for i, s in enumerate(response.source_nodes, 1):
    node = s.node
    text = node.get_content(metadata_mode=MetadataMode.NONE)
    meta = node.metadata or {}

    print(f"\nSource #{i}")
    print("-" * 100)
    print(f"Score      : {s.score:.4f}" if s.score is not None else "Score      : None")
    print(f"Node ID    : {node.node_id}")
    print(f"File       : {meta.get('file_name', 'N/A')}")
    print(f"Metadata   : {meta}")
    print("\nText Preview:")
    print(text[:1000], "...\n")



FINAL RESPONSE
The purpose of the encoder and decoder in the transformer network is to map an input sequence of symbol representations to a sequence of continuous representations, and then generate an output sequence of symbols one element at a time. The encoder takes in a sequence of input symbols and outputs a sequence of continuous representations, while the decoder takes these representations and generates the output sequence. Essentially, the encoder generates a continuous representation of the input sequence, and the decoder uses this representation to generate an output sequence, such as a translated sentence.

SOURCE NODES

Source #1
----------------------------------------------------------------------------------------------------
Score      : 0.7714
Node ID    : 113bde48-28b2-42a7-ab46-69c39f4cb577
File       : attention_is_all_you_need.pdf
Metadata   : {'page_label': '3', 'file_name': 'attention_is_all_you_need.pdf', 'file_path': '/content/docs/attention_is_all_you_need.pd